# 03 - Merge de Datasets: DETER + BDQueimadas
### Proyecto BIODIVERSITY-GUARD-Predict (1ACC0057 - Machine Learning)

## CORRECCION IMPORTANTE (v3): la clave de merge es `(municipio_norm, uf)`, no solo `municipio_norm`

Se detecto que Brasil tiene municipios con el mismo nombre en distintos estados (ej. `PAU D'ARCO` existe en Para y en Tocantins). Usar solo `municipio_norm` como clave de cruce mezcla datos de dos ciudades distintas bajo un mismo nombre -> contaminacion silenciosa de las variables climaticas para esos casos. La correccion es agregar siempre `uf` a la clave de todo `groupby` y todo `merge` de este notebook.

BDQueimadas trae el estado como nombre completo (`PARA`, `TOCANTINS`...); DETER lo trae como abreviatura (`PA`, `TO`...). Se construye un mapeo para unificarlos antes de cualquier cruce.

## Arquitectura de este notebook

Los notebooks 01 y 02 solo limpian datos **a nivel de registro individual**, sin agregar por semana ni por dia. Esa decision se toma aqui, porque cada frente necesita una granularidad distinta:

| | `dataset_alertas` (Frente 2 - Clasificacion) | `dataset_semanal` (Frente 1 - Regresion) |
|---|---|---|
| Una fila = | Una alerta individual de DETER | Una semana + un municipio (agregado) |
| Target | `Nivel_Riesgo_Amenaza` | `area_ha_total` |
| Ventana climatica usada | 7 dias previos a la fecha exacta de la alerta (sin incluir el futuro) | La semana calendario completa |
| Por que esta ventana | Usar "toda la semana" para una alerta de un dia especifico incluiria dias posteriores a la alerta -> fuga de informacion del futuro (data leakage) | Cada fila ya representa la semana completa como unidad, no hay leakage al usar el clima de esa misma semana |

## Archivos de entrada requeridos
1. `deter_limpio_merge.csv` - DETER, una fila = una alerta (snapshot congelado)
2. `bdqueimadas_limpio.csv` - BDQueimadas, una fila = un foco de calor

In [ ]:
import pandas as pd
import numpy as np

## Paso 1: Cargar los 2 archivos limpios y unificar el codigo de estado (`uf`)

In [ ]:
df_deter = pd.read_csv("deter_limpio_merge.csv")
df_deter["view_date"] = pd.to_datetime(df_deter["view_date"])

df_queimadas = pd.read_csv("bdqueimadas_limpio.csv")
df_queimadas["fecha"] = pd.to_datetime(df_queimadas["fecha"])

# BDQueimadas trae "Estado" como nombre completo -> mapear a la abreviatura que usa DETER en "uf"
mapa_uf = {
    "ACRE": "AC", "AMAPÁ": "AP", "AMAZONAS": "AM", "MARANHÃO": "MA",
    "MATO GROSSO": "MT", "PARÁ": "PA", "RONDÔNIA": "RO",
    "RORAIMA": "RR", "TOCANTINS": "TO"
}
df_queimadas["uf"] = df_queimadas["Estado"].map(mapa_uf)

print("Estados de BDQueimadas sin mapear (debe ser 0):", df_queimadas["uf"].isna().sum())

print("DETER (nivel alerta):", df_deter.shape)
print("BDQueimadas (nivel registro):", df_queimadas.shape)

# verificacion: cuantos municipios de DETER tienen nombre ambiguo entre estados
check = df_deter.groupby("municipio_norm")["uf"].nunique()
print("\nMunicipios de DETER con nombre ambiguo entre estados:", (check > 1).sum())
print(check[check > 1])

Estados de BDQueimadas sin mapear (debe ser 0): 0
DETER (nivel alerta): (49971, 19)
BDQueimadas (nivel registro): (354018, 11)

Municipios de DETER con nombre ambiguo entre estados: 1
municipio_norm
PAU D'ARCO    2
Name: uf, dtype: int64


## Paso 2: Dataset A - `dataset_alertas` (Frente 2 - Clasificacion)

### 2.1 Agregado DIARIO de BDQueimadas (por dia + municipio + estado)

In [ ]:
queimadas_diario = (
    df_queimadas
    .groupby(["fecha", "municipio_norm", "uf"])
    .agg(
        num_incendios=("FRP", "count"),
        frp_total=("FRP", "sum"),
        precipitacion_promedio=("Precipitacao", "mean"),
        dias_sin_lluvia_promedio=("DiaSemChuva", "mean"),
        riesgo_fuego_promedio=("RiscoFogo", "mean"),
    )
    .reset_index()
)

print(queimadas_diario.shape)
queimadas_diario.head()

(56664, 8)


,fecha,municipio_norm,uf,num_incendios,frp_total,precipitacion_promedio,dias_sin_lluvia_promedio,riesgo_fuego_promedio
0,2022-01-01,BOCA DO ACRE,AM,1,6.7,0.40,4.0,0.0
1,2022-01-01,CANUTAMA,AM,1,10.4,0.30,3.0,0.0
2,2022-01-01,GUAJARA-MIRIM,RO,2,23.8,0.15,8.0,0.4
3,2022-01-01,LABREA,AM,2,17.0,0.00,4.5,0.0
4,2022-01-01,NOVA MAMORE,RO,1,23.2,0.00,3.0,0.1


### 2.2 Grilla completa de fechas x (municipio, estado)

Necesaria para que la ventana movil no tenga huecos. Se construye a partir de los pares `(municipio_norm, uf)` unicos de DETER, no solo de `municipio_norm`, para no perder la distincion entre municipios homonimos.

In [ ]:
pares_municipio_uf = df_deter[["municipio_norm", "uf"]].drop_duplicates()
rango_fechas = pd.date_range(df_queimadas["fecha"].min(), df_queimadas["fecha"].max(), freq="D")

grilla = pares_municipio_uf.merge(pd.Series(rango_fechas, name="fecha"), how="cross")

queimadas_completo = grilla.merge(queimadas_diario, on=["fecha", "municipio_norm", "uf"], how="left")

queimadas_completo[["num_incendios", "frp_total"]] = queimadas_completo[["num_incendios", "frp_total"]].fillna(0)
# precipitacion_promedio, dias_sin_lluvia_promedio, riesgo_fuego_promedio: se dejan NaN

print(queimadas_completo.shape)

(465800, 8)


### 2.3 Ventana movil de 7 dias hacia atras (por municipio + estado)

In [ ]:
queimadas_completo = queimadas_completo.sort_values(["municipio_norm", "uf", "fecha"]).set_index("fecha")

cols_suma = ["num_incendios", "frp_total"]
cols_promedio = ["precipitacion_promedio", "dias_sin_lluvia_promedio", "riesgo_fuego_promedio"]

roll_suma = (queimadas_completo.groupby(["municipio_norm", "uf"])[cols_suma]
             .rolling("7D", min_periods=1).sum()
             .reset_index())

roll_prom = (queimadas_completo.groupby(["municipio_norm", "uf"])[cols_promedio]
             .rolling("7D", min_periods=1).mean()
             .reset_index())

queimadas_rolling = roll_suma.merge(roll_prom, on=["municipio_norm", "uf", "fecha"])
queimadas_rolling = queimadas_rolling.rename(
    columns={c: f"{c}_7d_previo" for c in cols_suma + cols_promedio}
)

print(queimadas_rolling.shape)
queimadas_rolling.head()

(465800, 8)


,municipio_norm,uf,fecha,num_incendios_7d_previo,frp_total_7d_previo,precipitacion_promedio_7d_previo,dias_sin_lluvia_promedio_7d_previo,riesgo_fuego_promedio_7d_previo
0,ABAETETUBA,PA,2022-01-01,0.0,0.0,NaN,NaN,NaN
1,ABAETETUBA,PA,2022-01-02,0.0,0.0,NaN,NaN,NaN
2,ABAETETUBA,PA,2022-01-03,0.0,0.0,NaN,NaN,NaN
3,ABAETETUBA,PA,2022-01-04,0.0,0.0,NaN,NaN,NaN
4,ABAETETUBA,PA,2022-01-05,0.0,0.0,NaN,NaN,NaN


### 2.4 Unir la ventana movil a cada alerta individual (por fecha exacta + municipio + estado)

In [ ]:
df_deter["fecha"] = df_deter["view_date"]

dataset_alertas = df_deter.merge(
    queimadas_rolling,
    on=["fecha", "municipio_norm", "uf"],
    how="left"
)

print("dataset_alertas:", dataset_alertas.shape)
assert dataset_alertas.shape[0] == df_deter.shape[0], "ALERTA: el merge cambio el numero de filas"

print("\nNivel_Riesgo_Amenaza se conserva intacto:")
print(dataset_alertas["Nivel_Riesgo_Amenaza"].value_counts())

cols_check = [c for c in dataset_alertas.columns if c.endswith("_7d_previo")]
print("\nNulos en variables climaticas 7d:")
print(dataset_alertas[cols_check].isnull().sum())

dataset_alertas: (49971, 25)

Nivel_Riesgo_Amenaza se conserva intacto:
Nivel_Riesgo_Amenaza
Moderado    32440
Alto         7743
Bajo         6483
Crítico      3305
Name: count, dtype: int64

Nulos en variables climaticas 7d:
num_incendios_7d_previo                  0
frp_total_7d_previo                      0
precipitacion_promedio_7d_previo      9012
dias_sin_lluvia_promedio_7d_previo    9024
riesgo_fuego_promedio_7d_previo       9057
dtype: int64


## Paso 3: Dataset B - `dataset_semanal` (Frente 1 - Regresion)

### 3.1 Agregado SEMANAL de DETER (por semana + municipio + estado)

In [ ]:
df_deter["semana"] = df_deter["view_date"].dt.to_period("W").dt.start_time

deter_semanal = (
    df_deter
    .groupby(["semana", "municipio_norm", "uf"])
    .agg(
        num_alertas=("area_ha", "count"),
        area_ha_total=("area_ha", "sum"),
        score_riesgo_promedio=("score_riesgo", "mean"),
    )
    .reset_index()
)

print(deter_semanal.shape)
deter_semanal.head()

(11445, 6)


,semana,municipio_norm,uf,num_alertas,area_ha_total,score_riesgo_promedio
0,2021-12-27,BOCA DO ACRE,AM,1,11.65,4.0
1,2021-12-27,CANDEIAS DO JAMARI,RO,2,48.69,4.0
2,2021-12-27,MANOEL URBANO,AC,2,33.45,3.5
3,2021-12-27,PORTO VELHO,RO,5,131.47,4.6
4,2021-12-27,SENA MADUREIRA,AC,2,38.57,5.5


### 3.2 Agregado SEMANAL de BDQueimadas (por semana + municipio + estado)

In [ ]:
df_queimadas["semana"] = df_queimadas["fecha"].dt.to_period("W").dt.start_time

queimadas_semanal = (
    df_queimadas
    .groupby(["semana", "municipio_norm", "uf"])
    .agg(
        num_incendios=("FRP", "count"),
        frp_total=("FRP", "sum"),
        frp_promedio=("FRP", "mean"),
        precipitacion_promedio=("Precipitacao", "mean"),
        dias_sin_lluvia_promedio=("DiaSemChuva", "mean"),
        riesgo_fuego_promedio=("RiscoFogo", "mean"),
    )
    .reset_index()
)

print(queimadas_semanal.shape)
queimadas_semanal.head()

(27130, 9)


,semana,municipio_norm,uf,num_incendios,frp_total,frp_promedio,precipitacion_promedio,dias_sin_lluvia_promedio,riesgo_fuego_promedio
0,2021-12-27,AMAJARI,RR,2,25.5,12.75,0.25,2.5,0.1
1,2021-12-27,BOCA DO ACRE,AM,1,6.7,6.70,0.40,4.0,0.0
2,2021-12-27,BONFIM,RR,2,30.7,15.35,0.00,3.0,0.5
3,2021-12-27,CANARANA,MT,1,5.3,5.30,0.00,0.0,0.0
4,2021-12-27,CANUTAMA,AM,1,10.4,10.40,0.30,3.0,0.0


### 3.3 Unir ambos agregados semanales (por semana + municipio + estado)

In [ ]:
dataset_semanal = deter_semanal.merge(
    queimadas_semanal,
    on=["semana", "municipio_norm", "uf"],
    how="left"
)

print("dataset_semanal:", dataset_semanal.shape)
assert dataset_semanal.shape[0] == deter_semanal.shape[0], "ALERTA: el merge cambio el numero de filas"

print("\nNulos tras el merge:")
print(dataset_semanal.isnull().sum())

dataset_semanal: (11445, 12)

Nulos tras el merge:
semana                         0
municipio_norm                 0
uf                             0
num_alertas                    0
area_ha_total                  0
score_riesgo_promedio          0
num_incendios               3621
frp_total                   3621
frp_promedio                3621
precipitacion_promedio      3621
dias_sin_lluvia_promedio    3630
riesgo_fuego_promedio       3657
dtype: int64


**Nota para el Modelado (Hito 3, no ahora):** si el objetivo es predecir `area_ha_total` a horizonte de 7/30 dias hacia adelante, las variables climaticas usadas como *features* deben ir *lageadas* (clima de la semana t-1, t-2... para predecir el area de la semana t), no el clima de la misma semana que se quiere predecir.

## Paso 4: Tratamiento de nulos (decision documentada)

| Columna | Significado del NaN | Tratamiento |
|---|---|---|
| `num_incendios`, `frp_total`, `frp_promedio` (y sus versiones `_7d_previo`) | Cero focos de calor detectados en esa ventana/periodo (hecho real y valido) | Rellenar con `0` |
| `precipitacion_promedio`, `dias_sin_lluvia_promedio`, `riesgo_fuego_promedio` (y `_7d_previo`) | BDQueimadas solo reporta clima **adjunto a un foco detectado**; sin foco no hay lectura, no significa "no llovio" | Se deja como `NaN` por ahora, pendiente de SISAM/NASA POWER (Notebooks 04/05)

In [ ]:
for col in ["num_incendios_7d_previo", "frp_total_7d_previo"]:
    dataset_alertas[col] = dataset_alertas[col].fillna(0)

for col in ["num_incendios", "frp_total", "frp_promedio"]:
    dataset_semanal[col] = dataset_semanal[col].fillna(0)

print("Nulos restantes en dataset_alertas:")
print(dataset_alertas.isnull().sum()[dataset_alertas.isnull().sum() > 0])
print("\nNulos restantes en dataset_semanal:")
print(dataset_semanal.isnull().sum()[dataset_semanal.isnull().sum() > 0])

Nulos restantes en dataset_alertas:
uc                                    46466
precipitacion_promedio_7d_previo       9012
dias_sin_lluvia_promedio_7d_previo     9024
riesgo_fuego_promedio_7d_previo        9057
dtype: int64

Nulos restantes en dataset_semanal:
precipitacion_promedio      3621
dias_sin_lluvia_promedio    3630
riesgo_fuego_promedio       3657
dtype: int64


## Paso 5: Guardar los datasets finales

In [ ]:
dataset_alertas = dataset_alertas.drop(columns=["fecha"])

dataset_alertas.to_csv("dataset_alertas_frente2.csv", index=False, encoding="utf-8")
dataset_semanal.to_csv("dataset_semanal_frente1.csv", index=False, encoding="utf-8")

print("Guardado: dataset_alertas_frente2.csv ->", dataset_alertas.shape)
print("Guardado: dataset_semanal_frente1.csv ->", dataset_semanal.shape)

Guardado: dataset_alertas_frente2.csv -> (49971, 24)
Guardado: dataset_semanal_frente1.csv -> (11445, 12)


## Pendiente (Notebooks 04/05)

- **SISAM** (PM2.5, PM10, O3, NO2, SO2, CO): agregar con la misma logica de doble granularidad Y la misma correccion de clave `(municipio_norm, uf)`.
- **NASA POWER** (temperatura, humedad, viento): opcional, misma logica si se agrega.